<a href="https://colab.research.google.com/github/ajnushka4128-code/hbb-gene-analyser/blob/main/HBB_Gene_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%%writefile requirements.txt
biopython

Writing requirements.txt
Overwriting requirements.txt


In [4]:
# Install Biopython if not already installed
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.3 MB/s eta 0:00:00


In [5]:
# Import necessary modules
import os
import io
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction, MeltingTemp as mt
import zipfile # Import the zipfile module

In [2]:
from google.colab import files
uploaded = files.upload()

Saving HBB_human_data to HBB_human_data


In [6]:
filename = list(uploaded.keys())[0]  # automatically grabs the file you just uploaded
print(f"Using file: {filename}")

# Extract if it's a zip (NCBI "datasets" downloads come as a zip)
if zipfile.is_zipfile(filename):
    print(f"'{filename}' is a zip file. Extracting contents...")
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('.')

def load_fasta(path):
    """Loads a FASTA file, stripping any junk lines before the first '>' header,
    and returns a list of SeqRecords (a file can contain more than one)."""
    with open(path, "r", encoding="latin-1") as handle:
        lines = handle.readlines()
    start = next((i for i, line in enumerate(lines) if line.startswith(">")), 0)
    lines = lines[start:]
    cleaned = io.StringIO("".join(lines))
    return list(SeqIO.parse(cleaned, "fasta-pearson"))

# The NCBI "datasets" package gives you all of these, each meaning something different
data_dir = "ncbi_dataset/data"
file_map = {
    "gene": "gene.fna",     # full genomic gene, includes introns/UTRs
    "rna":  "rna.fna",      # mature mRNA (introns spliced out)
    "cds":  "cds.fna",      # just the coding sequence — use this for translation/mutation checks
    "protein": "protein.faa" # the reference protein itself
}

records_by_type = {}
for label, fname in file_map.items():
    path = os.path.join(data_dir, fname)
    if os.path.exists(path):
        recs = load_fasta(path)
        records_by_type[label] = recs
        print(f"Loaded {label}: {len(recs)} record(s) from {fname}")
        for r in recs:
            print(f"   {r.id} — {len(r.seq)} bp — {r.description}")
    else:
        print(f"({label} not found — expected at {path})")

Using file: HBB_human_data
'HBB_human_data' is a zip file. Extracting contents...
Loaded gene: 2 record(s) from gene.fna
   NC_000011.10:c5227071-5225464 — 1608 bp — NC_000011.10:c5227071-5225464 HBB [organism=Homo sapiens] [GeneID=3043] [chromosome=11]
   NC_060935.1:c5286439-5284832 — 1608 bp — NC_060935.1:c5286439-5284832 HBB [organism=Homo sapiens] [GeneID=3043] [chromosome=11]
Loaded rna: 1 record(s) from rna.fna
   NM_000518.5 — 628 bp — NM_000518.5 HBB [organism=Homo sapiens] [GeneID=3043]
Loaded cds: 1 record(s) from cds.fna
   NM_000518.5:51-494 — 444 bp — NM_000518.5:51-494 HBB [organism=Homo sapiens] [GeneID=3043] [region=cds]
Loaded protein: 1 record(s) from protein.faa
   NP_000509.1 — 147 bp — NP_000509.1 HBB [organism=Homo sapiens] [GeneID=3043]


In [7]:
def show_basic_info(record):
    """
    Displays basic information about a Bio.SeqRecord object.

    Args:
        record (Bio.SeqRecord.SeqRecord): The sequence record to analyse.
    """
    print(f"ID: {record.id}")
    print(f"Name: {record.name}")
    print(f"Description: {record.description}")
    print(f"Length: {len(record.seq)} bp")
    if record.features:
        print(f"Number of annotated features: {len(record.features)}")
    else:
        print("No annotated features found for this record.")

def show_gc_content(record):
    """
    Calculates and displays the GC content of a sequence.

    Args:
        record (Bio.SeqRecord.SeqRecord): The sequence record to analyse.
    """
    gc = gc_fraction(record.seq) * 100
    print(f"GC content: {gc:.2f}%")
    print(f"AT content: {100 - gc:.2f}%")

def show_nucleotide_counts(record):
    """
    Counts and displays the proportion of each nucleotide (A, C, G, T) in a sequence.

    Args:
        record (Bio.SeqRecord.SeqRecord): The sequence record to analyse.
    """
    seq = record.seq.upper()
    total_length = len(seq)
    if total_length == 0:
        print("Sequence is empty, cannot calculate nucleotide counts.")
        return

    print("Nucleotide Composition:")
    for base in "ACGT":
        count = seq.count(base)
        print(f"{base}: {count} ({count/total_length*100:.2f}%)")
    # Handle non-ACGT characters if any, though less common with clean genomic data
    other_bases = 0
    for base in seq:
        if base not in "ACGT":
            other_bases += 1
    if other_bases > 0:
        print(f"Other/Ambiguous bases: {other_bases} ({other_bases/total_length*100:.2f}%)")

def show_transcription(record):
    """
    Transcribes a DNA sequence to its corresponding mRNA sequence.

    Args:
        record (Bio.SeqRecord.SeqRecord): The DNA sequence record to transcribe.
    """
    try:
        mrna = record.seq.transcribe()
        print("First 100 bases of mRNA (5' to 3'):")
        print(mrna[:100])
    except ValueError as e:
        print(f"Error during transcription: {e}. Ensure the sequence is DNA.")

def show_translation(record):
    """
    Translates a DNA/mRNA sequence into its corresponding protein sequence.

    Args:
        record (Bio.SeqRecord.SeqRecord): The sequence record to translate.
    """
    try:
        # to_stop=True ensures translation stops at the first encountered stop codon
        protein = record.seq.translate(to_stop=True, cds=False) # cds=False to avoid assuming CDS features
        print("First 100 amino acids of protein:")
        print(protein[:100])
        print(f"\nTotal protein length (to first stop codon): {len(protein)} aa")
    except ValueError as e:
        print(f"Error during translation: {e}. Ensure the sequence is coding and in frame.")

def show_reverse_complement(record):
    """
    Generates and displays the reverse complement of a DNA sequence.

    Args:
        record (Bio.SeqRecord.SeqRecord): The DNA sequence record.
    """
    rc = record.seq.reverse_complement()
    print("First 100 bases of reverse complement (5' to 3'):")
    print(rc[:100])

def find_orfs(record, min_len=100):
    """
    Finds and displays Open Reading Frames (ORFs) in a DNA sequence.
    ORFs are translated in all six frames (three on each strand).

    Args:
        record (Bio.SeqRecord.SeqRecord): The DNA sequence record.
        min_len (int): Minimum amino acid length for an ORF to be reported.
    """
    print(f"Searching for ORFs with minimum length {min_len} amino acids...")
    # Note: This function translates the entire record.seq. For accurate protein coding
    # region (CDS) prediction, you might need to extract CDS features if available
    # or use more advanced gene prediction tools.
    seq = record.seq
    orfs = []
    for strand, nuc in [(+1, seq), (-1, seq.reverse_complement())]:
        for frame in range(3):
            # Calculate length for translation to ensure it's a multiple of 3
            length = 3 * ((len(nuc) - frame) // 3)
            if length > 0:
                # Translate segment, handling potential non-standard start codons or partial codons
                trans = nuc[frame:frame+length].translate(table=1, to_stop=False, gap='-') # Use table 1 for standard code
                aa_start = 0
                # Split by stop codons represented as '*'
                for part in str(trans).split("*"):
                    if len(part) >= min_len and '-' not in part: # Exclude ORFs with gaps
                        orfs.append((strand, frame, aa_start, part))
                    aa_start += len(part) + 1 # +1 to account for the stop codon

    if not orfs:
        print(f"No ORFs found with minimum length {min_len} aa.")
    else:
        print(f"Found {len(orfs)} ORF(s):")
        for strand, frame, start_aa_pos, orf in orfs:
            print(f"  Strand {strand:+d}, Frame {frame}, Start amino acid position {start_aa_pos}, Length {len(orf)} aa")
            print(f"    {orf[:60]}{'...' if len(orf) > 60 else ''}")

def search_motif(record, motif):
    """
    Searches for a specific DNA motif (subsequence) within the record's sequence.

    Args:
        record (Bio.SeqRecord.SeqRecord): The sequence record to search.
        motif (str): The nucleotide motif to search for (case-insensitive).
    """
    if not motif:
        print("Please provide a motif to search.")
        return

    seq = str(record.seq).upper()
    motif = motif.upper()
    positions = []
    start = 0
    while True:
        idx = seq.find(motif, start)
        if idx == -1:
            break
        positions.append(idx)
        start = idx + 1 # Move past the found motif to find subsequent occurrences

    if positions:
        print(f"Motif '{motif}' found at {len(positions)} position(s) (0-indexed):")
        # Display up to 20 positions for brevity
        print(f"  {positions[:20]}{'...' if len(positions) > 20 else ''}")
    else:
        print(f"Motif '{motif}' not found in the sequence.")

def check_sickle_cell_mutation(record):
    """
    Checks for the classic sickle-cell mutation (GAG -> GTG) at codon 6 of the HBB gene.
    This mutation changes Glutamic acid to Valine in the beta-globin protein.

    IMPORTANT NOTE:
    This check is highly dependent on the input sequence starting exactly at the
    translation start site (ATG) of the coding sequence (CDS) for HBB.
    If the record.seq includes introns, UTRs, or is not aligned correctly to the CDS,
    this check will not be scientifically accurate or reliable.
    It serves as an illustrative example of how to check for a specific point mutation.

    Args:
        record (Bio.SeqRecord.SeqRecord): The HBB sequence record to check.
    """
    seq_str = str(record.seq).upper()
    # Codon 6 starts at index 15 (0-indexed) if the sequence is the CDS
    # 5 codons * 3 bases/codon = 15 bases before codon 6
    if len(seq_str) < 21: # Need at least 6 codons (18 bases)
        print("Sequence is too short to check codon 6.")
        return

    codon6 = seq_str[18:21]
    print(f"Codon 6 in this sequence (assuming CDS start at index 0): {codon6}")

    if codon6 == "GAG":
        print("  -> Matches normal Hemoglobin A (Glutamic acid).")
    elif codon6 == "GTG":
        print("  -> Matches sickle-cell mutation Hemoglobin S (Valine).")
    else:
        print("  -> Codon 6 does not directly match GAG or GTG. This check assumes ")
        print("     the sequence is the exact coding sequence (CDS) for accurate interpretation.")

def melting_temperature(record, length=50):
    """
    Estimates the melting temperature (Tm) for a fragment of the sequence.
    Uses the Wallace rule, a simple estimation method.

    Args:
        record (Bio.SeqRecord.SeqRecord): The DNA sequence record.
        length (int): The length of the sequence fragment from the start to use for Tm calculation.
    """
    if len(record.seq) < length:
        print(f"Sequence is too short ({len(record.seq)} bp) to calculate Tm for a {length} bp fragment.")
        return

    fragment = record.seq[:length] # Take the first 'length' bases
    try:
        tm = mt.Tm_Wallace(str(fragment)) # Tm_Wallace expects a string
        print(f"Estimated melting temperature (Wallace rule) for first {length} bp: {tm:.2f} °C")
    except Exception as e:
        print(f"Error calculating melting temperature: {e}. Ensure the fragment contains only valid DNA bases (A, C, G, T).")

In [ ]:
SEQUENCE_INFO = {
    "gene": {
        "explain": "Full genomic sequence, including introns and regulatory regions (not just the protein-coding parts).",
        "good_for": "Basic composition tools: GC content, nucleotide counts, reverse complement, motif search.",
        "avoid_for": "Translation and the sickle-cell check — these need pure coding sequence, and introns will make the results meaningless."
    },
    "rna": {
        "explain": "The mature mRNA — introns already removed, but still includes the untranslated regions (UTRs) before and after the actual protein-coding part.",
        "good_for": "ORF finding (a good demo of detecting the coding region from a longer sequence), composition tools.",
        "avoid_for": "Direct translation may include some UTR noise before hitting the real start codon."
    },
    "cds": {
        "explain": "Just the protein-coding sequence (CDS) — starts exactly at the ATG start codon, ends at the stop codon. No introns, no UTRs.",
        "good_for": "Translation, and the sickle-cell mutation check. Use this one for anything involving the actual protein.",
        "avoid_for": "Nothing — this is the most reliable choice for most tools."
    },
    "protein": {
        "explain": "The amino acid sequence itself, not DNA.",
        "good_for": "Comparing by eye against what the CDS translates to.",
        "avoid_for": "DNA-only tools: transcription, translation, reverse complement, sickle-cell check, melting temperature. These will error or be meaningless on a protein sequence."
    }
}

def choose_record():
    print("Which sequence would you like to analyse?\n")
    options = list(records_by_type.keys())
    for i, label in enumerate(options, start=1):
        info = SEQUENCE_INFO.get(label, {})
        print(f"{i}. {label.upper()} ({file_map[label]})")
        print(f"   What it is:   {info.get('explain', '')}")
        print(f"   Good for:     {info.get('good_for', '')}")
        print(f"   Avoid for:    {info.get('avoid_for', '')}\n")

    while True:
        choice = input("Enter number: ").strip()
        if choice.isdigit() and 1 <= int(choice) <= len(options):
            label = options[int(choice) - 1]
            recs = records_by_type[label]
            if len(recs) == 1:
                return recs[0]
            else:
                print(f"\n'{label}' has multiple records (likely different genome assemblies):")
                for i, r in enumerate(recs, start=1):
                    print(f"  {i}. {r.id} — {r.description}")
                sub = input("Enter number: ").strip()
                if sub.isdigit() and 1 <= int(sub) <= len(recs):
                    return recs[int(sub) - 1]
        print("Invalid choice, try again.")

def show_menu():
    print("\n" + "="*50)
    print(" HBB Gene Analyser — Menu")
    print("="*50)
    print("1. Show basic sequence info")
    print("2. GC content")
    print("3. Nucleotide composition (A/C/G/T counts)")
    print("4. Transcribe DNA -> mRNA")
    print("5. Translate -> protein")
    print("6. Reverse complement")
    print("7. Find Open Reading Frames (ORFs)")
    print("8. Search for a motif/subsequence")
    print("9. Check sickle-cell mutation (codon 6)")
    print("10. Estimate melting temperature (Tm)")
    print("11. Switch sequence (gene / rna / cds / protein)")
    print("0. Exit")
    print("="*50)

def run_menu():
    record = choose_record()
    while True:
        print(f"\n[Currently analysing: {record.id} — {len(record.seq)} bp]")
        show_menu()
        choice = input("Choose an option: ").strip()

        if choice == "1":
            show_basic_info(record)
        elif choice == "2":
            show_gc_content(record)
        elif choice == "3":
            show_nucleotide_counts(record)
        elif choice == "4":
            show_transcription(record)
        elif choice == "5":
            show_translation(record)
        elif choice == "6":
            show_reverse_complement(record)
        elif choice == "7":
            try:
                min_len_input = input("Minimum ORF length in amino acids (default 100): ").strip()
                min_len = int(min_len_input) if min_len_input else 100
                find_orfs(record, min_len)
            except ValueError:
                print("Invalid input. Please enter an integer for minimum ORF length.")
        elif choice == "8":
            motif = input("Enter motif/sequence to search for: ").strip()
            search_motif(record, motif)
        elif choice == "9":
            check_sickle_cell_mutation(record)
        elif choice == "10":
            try:
                length_input = input("Fragment length in bp for Tm calculation (default 50): ").strip()
                length = int(length_input) if length_input else 50
                melting_temperature(record, length)
            except ValueError:
                print("Invalid input. Please enter an integer for fragment length.")
        elif choice == "11":
            record = choose_record()
        elif choice == "0":
            print("Exiting. Goodbye!")
            break
        else:
            print("Invalid choice, please try again with a number from the menu.")

run_menu()

Which sequence would you like to analyse?

1. GENE (gene.fna)
   What it is:   Full genomic sequence, including introns and regulatory regions (not just the protein-coding parts).
   Good for:     Basic composition tools: GC content, nucleotide counts, reverse complement, motif search.
   Avoid for:    Translation and the sickle-cell check — these need pure coding sequence, and introns will make the results meaningless.

2. RNA (rna.fna)
   What it is:   The mature mRNA — introns already removed, but still includes the untranslated regions (UTRs) before and after the actual protein-coding part.
   Good for:     ORF finding (a good demo of detecting the coding region from a longer sequence), composition tools.
   Avoid for:    Direct translation may include some UTR noise before hitting the real start codon.

3. CDS (cds.fna)
   What it is:   Just the protein-coding sequence (CDS) — starts exactly at the ATG start codon, ends at the stop codon. No introns, no UTRs.
   Good for:     Tran